# 02 — Feature Extraction

Produces two parallel feature views of the train/val/test split:

1. **Sparse TF-IDF** (1–2 grams, sublinear TF) — feeds the classical-ML tier.
2. **Dense sentence embeddings** (`all-MiniLM-L6-v2`) — feeds BERTopic and zero-shot similarity work.

`all-MiniLM-L6-v2` is chosen over `all-mpnet-base-v2` for CPU speed; swap if a GPU is available.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import numpy as np
import polars as pl
import joblib
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

from utils import ARTIFACTS_DIR, set_seed
set_seed()

In [ ]:
splits = {
    name: pl.read_parquet(ARTIFACTS_DIR / f"split_{name}.parquet")
    for name in ("train", "val", "test")
}
for n, s in splits.items():
    print(f"{n:5s} {s.shape}")

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents="unicode",
)
X_train = tfidf.fit_transform(splits["train"]["review"].to_list())
X_val   = tfidf.transform(splits["val"]["review"].to_list())
X_test  = tfidf.transform(splits["test"]["review"].to_list())
print("TF-IDF shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

sparse.save_npz(ARTIFACTS_DIR / "tfidf_train.npz", X_train)
sparse.save_npz(ARTIFACTS_DIR / "tfidf_val.npz",   X_val)
sparse.save_npz(ARTIFACTS_DIR / "tfidf_test.npz",  X_test)
joblib.dump(tfidf, ARTIFACTS_DIR / "tfidf_vectorizer.joblib")

In [ ]:
# Dense embeddings. Skip if sentence-transformers is unavailable;
# notebooks 06 and 07 will fall back to recomputing on the fly.
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    for name, df in splits.items():
        emb = model.encode(
            df["review"].to_list(),
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        np.save(ARTIFACTS_DIR / f"emb_{name}.npy", emb)
        print(f"  emb_{name}.npy -> {emb.shape}")
except ImportError:
    print("sentence-transformers not installed; skipping dense features.")

## Artifacts produced

- `tfidf_{train,val,test}.npz` + `tfidf_vectorizer.joblib`
- `emb_{train,val,test}.npy` (if sentence-transformers is installed)
